# Trening modelu na danych resting-state

Ten notebook trenuje model DeepMReye **od zera na danych resting-state** (29 podmiotów)
z użyciem niestandardowej funkcji straty łączącej **Smooth L1 (Huber)** z komponentem **korelacji Pearsona**.

**Podział danych resting-state:**
- 18 podmiotów — zbiór treningowy
- 5 podmiotów — zbiór walidacyjny (do monitorowania overfittingu podczas treningu)
- 6 podmiotów — zbiór testowy (wydzielony, nieużywany podczas treningu)


In [ ]:
import os 
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async" 
import tensorflow as tf 
tf.keras.backend.clear_session() 
import pandas as pd 
from deepmreye import analyse, architecture, preprocess, train 
from deepmreye.util import data_generator, model_opts, util 
import numpy as np 
from smooth_loss import train_model_with_smoothl1

gpus = tf.config.experimental.list_physical_devices('GPU') 
tf.config.experimental.set_memory_growth(gpus[0], True)


## Funkcje pomocnicze

`create_holdout_generators` opakowuje generator danych DeepMReye.
Przyjmuje jawne listy ścieżek `train_list` / `test_list` i zwraca generatory treningowe,
testowe oraz per-podmiot używane podczas ewaluacji.

In [ ]:
def create_holdout_generators(datasets=None, train_split=0.6, train_list=None, test_list=None, **args):
    if train_list is not None and test_list is not None:
        full_training_list = train_list
        full_testing_list = test_list
    else:
        full_training_list, full_testing_list = list(), list()
        for fn_data in datasets:
            this_file_list = [fn_data + p for p in os.listdir(fn_data)]
            np.random.shuffle(this_file_list)
            split_idx = int(train_split * len(this_file_list))
            this_training_list = this_file_list[0:split_idx]
            this_testing_list = this_file_list[split_idx:]
            full_training_list.extend(this_training_list)
            full_testing_list.extend(this_testing_list)

    (
        training_generator,
        testing_generator,
        single_testing_generators,
        single_testing_names,
        single_training_generators,
        single_training_names,
    ) = data_generator.create_generators(full_training_list, full_testing_list, **args)

    return (
        training_generator,
        testing_generator,
        single_testing_generators,
        single_testing_names,
        single_training_generators,
        single_training_names,
        full_testing_list,
        full_training_list,
    )

## Domyślna konfiguracja modelu

Wczytywane są domyślne hiperparametry DeepMReye (`model_opts.get_opts()`).
Zostaną one nadpisane w dalszej części notebooka wartościami znalezionymi przez Ray Tune.

In [ ]:
opts = model_opts.get_opts()
print(opts)

## Wczytywanie plików resting-state

Plik `.npz` każdego podmiotu zbierany jest z folderu `dataset7_resting_state`.
Zbiór zawiera 29 podmiotów oznaczonych identyfikatorami `9xxx`.

In [ ]:
all_files=[]
for root, dirs, files in os.walk('/mnt/compneuro/deepmreye_finetuning/processed_data/dataset7_resting_state/'):
    for name in files:
        all_files.append(os.path.join(root,name))
        print(os.path.join(root,name))

In [ ]:
datasets = {}
for f in all_files:
    dataset_name = os.path.basename(os.path.dirname(f))
    datasets.setdefault(dataset_name, []).append(f)

train_list_rs, test_list_rs = [], []
for dataset_name, files in datasets.items():
    files = sorted(files)                
    np.random.shuffle(files)            
    split_idx = int(0.8 * len(files))    
    train_list_rs.extend(files[:split_idx])
    test_list_rs.extend(files[split_idx:])

## Podział na zbiory treningowy, walidacyjny i testowy

Stały podział jest wczytywany z pliku `.txt` zapisanego wcześniej, co zapewnia **reprodukowalność** -
te same 6 podmiotów testowych jest używanych we wszystkich eksperymentach.

Zbiór treningowy (23 podmioty) jest dodatkowo dzielony:
- `train_list` — pierwsze 18 podmiotów (właściwy trening)
- `validation_list` — pozostałe 5 podmiotów (walidacja w trakcie treningu)


In [ ]:
train_list_rs=np.loadtxt('/mnt/compneuro/deepmreye_finetuning/Zosia/train_list_rs.txt',dtype=str)
test_list_rs=np.loadtxt('/mnt/compneuro/deepmreye_finetuning/Zosia/test_list_rs.txt',dtype=str)

train_list = train_list_rs[5:]
validation_list = train_list_rs[:5]
test_list = test_list_rs
train_list_rs = train_list

In [ ]:
print("Total train:", len(train_list_rs))
print("Fine tunning total:", len(validation_list))
print("Total test:", len(test_list))

## Konwersja ścieżek i tworzenie generatorów danych

Ścieżki względne są konwertowane na bezwzględne.
Generatory są tworzone z augmentacją danych. `mixed_batches=True` - każda paczka zawiera próbki z wielu podmiotów, co stabilizuje trening.

In [ ]:
train_list_rs_absolute = []
for f in train_list_rs:
    if f.startswith('./processed_data/'):
        absolute_path = f.replace('./processed_data/', '/mnt/compneuro/deepmreye_finetuning/processed_data/')
    else:
        absolute_path = f
    train_list_rs_absolute.append(absolute_path)

test_list_rs_absolute = []
for f in test_list_rs:
    if f.startswith('./processed_data/'):
        absolute_path = f.replace('./processed_data/', '/mnt/compneuro/deepmreye_finetuning/processed_data/')
    else:
        absolute_path = f
    test_list_rs_absolute.append(absolute_path)

In [ ]:
generators_rs = create_holdout_generators(
    train_list=train_list_rs_absolute,
    test_list=test_list_rs_absolute, 
    batch_size=opts['batch_size'],
    augment_list=(
        (opts['rotation_x'], opts['rotation_y'], opts['rotation_z']), 
        opts['shift'], 
        opts['zoom']
    ),
    mixed_batches=True
)

## Hiperparametry modelu

Parametry najlepszej konfiguracji znalezionej przez **Ray Tune** (30 prób):

| Parametr | Wartość |
|---|---|
| `lr` | 4.57e-05 |
| `smooth_l1_delta` | 0.05 |
| `loss_pearson` | 0.107 |
| `dropout_rate` | 0.25 |
| `gaussian_noise` | 0.05 |
| `loss_confidence` | 0.5 |
| `epochs` | 200 |

Funkcja straty: `α · SmoothL1 + (1−α) · (1 − Pearson_r)`, gdzie δ = 0.05.
Monte Carlo Dropout (`mc_dropout=True`) jest włączony podczas treningu dla regularyzacji.

In [ ]:
opts['num_fc'] = 256
#opts['num_dense'] = 1
opts['dropout_rate']=0.25
opts['mc_dropout'] = True
opts['gaussian_noise']=0.05
opts['epochs']=200
opts['loss_confidence'] = 0.5
opts['steps_per_epoch'] = 60
opts['validation_steps']=15
opts['lr']=4.567854407868889e-05
opts['smooth_l1_delta'] = 0.05
opts['loss_pearson'] = 0.10683831361427415

## Trening modelu

Model jest trenowany od zera wyłącznie na danych resting-state.
Używana jest niestandardowa funkcja `train_model_with_smoothl1` z modułu `smooth_loss`,
która implementuje stratę Smooth L1 + Pearson zamiast domyślnej straty euklidesowej DeepMReye.

In [ ]:
model, model_inference,history = train_model_with_smoothl1(
    dataset="resting_state_data_pearson_max",
    generators=generators_rs,
    opts=opts,                 
    is_resting_state=True,
    save=True,
    verbose=1
)

## Krzywa uczenia

Wykres pokazuje stratę treningową i walidacyjną w skali logarytmicznej po każdej epoce.
Rozbieżność między krzywą treningową a walidacyjną wskazuje na **overfitting**. Wykres jest widoczny w pliku `figures\training_and_validation_lines.png`

In [ ]:
import matplotlib.pyplot as plt

def plot_training_history(history):
    loss = history.history['smoothl1_loss']
    val_loss = history.history['val_smoothl1_loss']
    epochs = range(1, len(loss) + 1)

    plt.figure(figsize=(10, 6))
    plt.plot(epochs, loss, 'b-', label='Training Loss')
    plt.plot(epochs, val_loss, 'r-', label='Validation Loss')
    plt.yscale("log")
    plt.title('Learning Curve')
    plt.xlabel('Epochs')
    plt.ylabel('Smooth L1 Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

plot_training_history(history)

## Ewaluacja na zbiorze treningowym

Ewaluacja na 18 podmiotach treningowych służy do sprawdzenia, czy model poprawnie dopasował się do danych,
na których był trenowany.

Wyniki widoczne w `figures\2-Training_data_evaluation.png`

In [ ]:
generators_rs = data_generator.create_generators(train_list_rs_absolute,
                                                 train_list_rs_absolute)

generators_rs = (
    *generators_rs,
    train_list_rs_absolute,
    train_list_rs_absolute
)

In [ ]:
(evaluation_rs, scores_rs) = train.evaluate_model(
    dataset='resting_state',
    model=model_inference,
    generators=generators_rs,
    save=True,
    model_description='smoothl1_model',
    verbose=2
)

In [ ]:
fig = analyse.visualise_predictions_slider(
    evaluation_rs,
    scores_rs,
    color="rgb(0, 150, 175)",
    bg_color="rgb(255,255,255)",
    ylim=[-2, 2]
)
fig.show()

## Ewaluacja na zbiorze testowym

Ewaluacja na 6 wydzielonych podmiotach testowych daje **rzeczywistą miarę generalizacji** modelu.

Korelacja Pearsona r na zbiorze testowym jest znacznie niższa niż na treningowym - model overfittował i nie generalizuje na nowych podmiotach.

Wyniki widoczne w `figures\2-Test_data_evaluation.png`

In [ ]:
generators_rs = data_generator.create_generators(test_list_rs_absolute,
                                                 test_list_rs_absolute)

generators_rs = (
    *generators_rs,
    test_list_rs_absolute,
    test_list_rs_absolute
)

In [ ]:
(evaluation_rs, scores_rs) = train.evaluate_model(
    dataset='resting_state',
    model=model_inference,
    generators=generators_rs,
    save=True,
    model_description='smoothl1_model',
    verbose=2
)

In [ ]:
fig = analyse.visualise_predictions_slider(
    evaluation_rs,
    scores_rs,
    color="rgb(0, 150, 175)",
    bg_color="rgb(255,255,255)",
    ylim=[-2, 2]
)
fig.show()